<a href="https://colab.research.google.com/github/kili-technology/kili-python-sdk/blob/main/recipes/copy_workflow_between_projects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to copy a workflow between projects

In this recipe, we will learn how to replicate a multi-step workflow from one project to another using `copy_workflow_from_project`.

This is useful when you want to:
- Bootstrap a new project with the workflow configuration as an existing one
- Standardize workflow settings across several projects

The copy operation:
1. Reads all workflow steps from the **source** project (step type, consensus coverage, step coverage, send-back links…)
2. Replaces the steps in the **destination** project with a matching structure
3. Assigns all activated destination-project members to each new step (labelers for DEFAULT steps, non-labelers for REVIEW steps)
4. Remaps any `sendBackStepId` references to the newly created destination-step IDs

> **Requirements**
> - Both projects must use **workflow V2**.
> - The destination project must have **no labels** at the time of copying.
> - If the first source step has a consensus setting, the destination project must have activated members.

## Setup

In [ ]:
%pip install kili

In [ ]:
from kili.client import Kili

kili = Kili(
    # api_endpoint="https://cloud.kili-technology.com/api/label/v2/graphql",
)

## Create the source project and its workflow

We start by creating a source project and configuring a two-step workflow:
- A **Labeling** step (DEFAULT) with 50 % consensus coverage and 2 required labelers
- A **Review** step covering 80 % of assets, that can send assets back to Labeling

In [ ]:
json_interface = {
    "jobs": {
        "JOB_0": {
            "mlTask": "CLASSIFICATION",
            "required": 0,
            "isChild": False,
            "content": {
                "categories": {
                    "CATEGORY_A": {"name": "Category A", "children": []},
                    "CATEGORY_B": {"name": "Category B", "children": []},
                },
                "input": "radio",
            },
            "isNew": False,
        }
    }
}

source_project_id = kili.create_project(
    input_type="IMAGE",
    json_interface=json_interface,
    title="[Kili SDK Notebook]: copy workflow — source",
)["id"]
print("Source project:", source_project_id)

Source project: cmn8z1aoz004rme4i4zx8c81w


Add members to the source project so that steps have assignees:

In [ ]:
# Replace with real email addresses that exist in your organisation
source_labeler_emails = ["labeler1@example.com", "labeler2@example.com"]
source_reviewer_emails = ["reviewer1@example.com"]

for email in source_labeler_emails:
    kili.append_to_roles(project_id=source_project_id, user_email=email, role="LABELER")

for email in source_reviewer_emails:
    kili.append_to_roles(project_id=source_project_id, user_email=email, role="REVIEWER")

Retrieve the member IDs needed to assign steps, then create the workflow:

In [ ]:
from kili.use_cases.project_workflow import ProjectWorkflowUseCases  # noqa: F401

# Fetch activated members and split by role
members = kili.project_users(project_id=source_project_id, fields=["role", "user.id"])
all_ids = [m["user"]["id"] for m in members]
reviewer_ids = [m["user"]["id"] for m in members if m["role"] != "LABELER"]

# Get the default (first) step so we can reference its id for sendBackStepId later
initial_steps = kili.get_steps(project_id=source_project_id)
first_step_id = initial_steps[0]["id"] if initial_steps else None

# Build a two-step workflow
kili.update_project_workflow(
    project_id=source_project_id,
    enforce_step_separation=True,
    update_steps=[
        {
            "id": first_step_id,
            "name": "Labeling Renamed",
            "consensus_coverage": 50,
            "number_of_expected_labels_for_consensus": 2,
        }
    ]
    if first_step_id
    else None,
    create_steps=[
        {
            "name": "Review 2",
            "type": "REVIEW",
            "step_coverage": 80,
            "assignees": reviewer_ids,
            "send_back_step_id": first_step_id,
        }
    ],
)

# Confirm the source workflow
source_steps = kili.get_steps(project_id=source_project_id)
print("Source workflow steps:")
for step in source_steps:
    print(" -", step)

Source workflow steps:
 - {'assignees': [{'email': 'test+edouard@kili-technology.com', 'id': 'user-2'}, {'email': 'labeler1@example.com', 'id': 'cmn8yvbve000lme4i2yxa252z'}, {'email': 'labeler2@example.com', 'id': 'cmn8yvbzi000tme4ifga589w7'}, {'email': 'reviewer1@example.com', 'id': 'cmn8yvc2h0011me4iajuh8v1r'}], 'type': 'DEFAULT', 'name': 'Labeling Renamed', 'id': 'cmn8z1apd004tme4i9bvbfwrv'}
 - {'assignees': [{'email': 'test+edouard@kili-technology.com', 'id': 'user-2'}, {'email': 'reviewer1@example.com', 'id': 'cmn8yvc2h0011me4iajuh8v1r'}], 'type': 'REVIEW', 'name': 'Review', 'id': 'cmn8z1apn004ume4i8hwm0kce'}
 - {'assignees': [{'email': 'test+edouard@kili-technology.com', 'id': 'user-2'}, {'email': 'test+max@kili-technology.com', 'id': 'user-7'}, {'email': 'test+queues@kili-technology.com', 'id': 'user-9'}, {'email': 'test+github@kili-technology.com', 'id': 'user-6'}, {'email': 'reviewer1@example.com', 'id': 'cmn8yvc2h0011me4iajuh8v1r'}], 'type': 'REVIEW', 'name': 'Review 2', 'id'

## Create the destination project

In [ ]:
destination_project_id = kili.create_project(
    input_type="IMAGE",
    json_interface=json_interface,
    title="[Kili SDK Notebook]: copy workflow — destination",
)["id"]
print("Destination project:", destination_project_id)

Destination project: cmn8z1hbg005yme4i3e6sc8mv


Add at least as many labelers to the destination project as required by the source workflow's consensus setting (2 in our case):

In [ ]:
# Replace with real email addresses that exist in your organisation
dest_labeler_emails = ["dest_labeler1@example.com", "dest_labeler2@example.com"]
dest_reviewer_emails = ["dest_reviewer1@example.com"]

for email in dest_labeler_emails:
    kili.append_to_roles(project_id=destination_project_id, user_email=email, role="LABELER")

for email in dest_reviewer_emails:
    kili.append_to_roles(project_id=destination_project_id, user_email=email, role="REVIEWER")

## Copy the workflow

A single call copies the full workflow configuration from the source to the destination project:

In [ ]:
result = kili.copy_workflow_from_project(
    destination_project_id=destination_project_id,
    source_project_id=source_project_id,
)
print(result)

{'enforceStepSeparation': True, 'steps': [{'id': 'cmn8z1hbw0060me4iek7parym'}, {'id': 'cmn8z1kio0070me4i7nrta6fc'}, {'id': 'cmn8z1kiu0071me4ie9kg0tn9'}]}


## Verify the copied workflow

The destination project should now have the same step structure as the source:

In [ ]:
dest_steps = kili.get_steps(
    project_id=destination_project_id,
    fields=[
        "steps.name",
        "steps.type",
        "steps.consensusCoverage",
        "steps.numberOfExpectedLabelsForConsensus",
        "steps.stepCoverage",
        "steps.sendBackStepId",
    ],
)
print("Destination workflow steps after copy:")
for step in dest_steps:
    print(" -", step)

Destination workflow steps after copy:
 - {'name': 'Labeling Renamed', 'type': 'DEFAULT', 'consensusCoverage': 50, 'numberOfExpectedLabelsForConsensus': 2, 'stepCoverage': None, 'sendBackStepId': None}
 - {'name': 'Review', 'type': 'REVIEW', 'consensusCoverage': None, 'numberOfExpectedLabelsForConsensus': None, 'stepCoverage': None, 'sendBackStepId': None}
 - {'name': 'Review 2', 'type': 'REVIEW', 'consensusCoverage': None, 'numberOfExpectedLabelsForConsensus': None, 'stepCoverage': 80, 'sendBackStepId': 'cmn8z1hbw0060me4iek7parym'}


You should see:
- A **Labeling** (DEFAULT) step with `consensusCoverage=50` and `numberOfExpectedLabelsForConsensus=2`
- A **Review** step with `stepCoverage=80` and a `sendBackStepId` pointing to the new Labeling step ID

The assignees on each step are the destination-project members, not the source-project members.

## Cleanup

In [ ]:
kili.delete_project(source_project_id)
kili.delete_project(destination_project_id)

'cmn8z0wln0044me4icsblhtja'